# Telecom Bronze Layer (Kafka Streaming)
Consumes real-time CDRs from Kafka topic `telecom_cdrs` and lands them as raw Delta tables in the Bronze layer.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StringType, IntegerType
from pyspark.sql.functions import from_json, col

print("Starting Spark Streaming Session...")
spark = SparkSession.builder \
    .appName("Telecom_Kafka_Bronze") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark Ready for Kafka!")

In [4]:
# Define the schema matching NiFi/Simulator's JSON output
json_schema = StructType() \
    .add("call_id", StringType()) \
    .add("customer_number", StringType()) \
    .add("tower_id", StringType()) \
    .add("event_type", StringType()) \
    .add("duration_seconds", IntegerType()) \
    .add("data_consumed_mb", IntegerType()) \
    .add("call_status", StringType()) \
    .add("latency_ms", IntegerType()) \
    .add("signal_strength_dbm", IntegerType()) \
    .add("network_condition", StringType()) \
    .add("timestamp", StringType())

# Read stream from Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "telecom_cdrs") \
    .option("startingOffsets", "earliest") \
    .load()

# Kafka stores message body in a binary 'value' column. Cast it to string and parse JSON.
df_parsed = df_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), json_schema).alias("data")) \
    .select("data.*")

print("Kafka Stream Connected!")

Kafka Stream Connected!


In [5]:
# Write the streaming data to MinIO Bronze layer as Delta Format
query = df_parsed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3://bronze/checkpoints/telecom_cdrs_raw") \
    .start("s3://bronze/telecom_cdrs_raw")

print("Streaming data is now being written to S3 Bronze layer. Leave this cell running!")
query.awaitTermination()

Streaming data is now being written to S3 Bronze layer. Leave this cell running!


StreamingQueryException: [STREAM_FAILED] Query [id = 5a743ab8-109d-4aad-b020-72807a3f5bfb, runId = 5cdb41bd-4dee-4716-873d-a60c20d5b79c] terminated with exception: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: describeTopics